# Atividade: Predição de Diabetes (Foco em Pré-processamento)
Este notebook está organizado para que a equipe possa realizar a limpeza e o tratamento dos dados antes de submeter os resultados ao servidor.

### 1. Importações e Configurações
Configuramos as bibliotecas e a chave de acesso da equipe.

In [5]:
import pandas as pd
import requests
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import accuracy_score, classification_report

#config
DEV_KEY = "2_computeiros_e_um_matemagico"
URL = "https://aydanomachado.com/mlclass/01_Preprocessing.php"

feature_cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
                'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age']

df_train = pd.read_csv('diabetes_dataset.csv')
df_app = pd.read_csv('diabetes_app.csv')

### 2. Carga dos Dados
Lemos os arquivos CSV originais.

In [6]:
df_train = pd.read_csv('diabetes_dataset.csv')
df_app = pd.read_csv('diabetes_app.csv')

print('Dados de treino carregados:', df_train.shape)
print('Dados de aplicação carregados:', df_app.shape)

Dados de treino carregados: (572, 9)
Dados de aplicação carregados: (196, 8)


### 3. Pré-processamento de Dados
**Nesta seção, vamos implementar as técnicas de pré-processamento**

In [7]:
#removendo as colunas com quase 50% de dados faltantes
colunas_para_remover = ['Insulin', 'SkinThickness']
df_train = df_train.drop(columns=colunas_para_remover)
df_app = df_app.drop(columns=colunas_para_remover)

features_atualizadas = ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']

X_bruto = df_train[features_atualizadas]
y_bruto = df_train['Outcome']
X_app_bruto = df_app[features_atualizadas]

#colunas que ainda precisam de preenchimento
colunas_com_vazios = ['Glucose', 'BloodPressure', 'BMI']

In [8]:
#simula local, sem data leak dessa vez
X_treino, X_teste, y_treino, y_teste = train_test_split(X_bruto, y_bruto, test_size=0.2, random_state=42)

#imputação com a mediana do Treino
medias_seguras = X_treino[colunas_com_vazios].median()

X_treino_limpo = X_treino.copy()
X_teste_limpo = X_teste.copy()

X_treino_limpo[colunas_com_vazios] = X_treino_limpo[colunas_com_vazios].fillna(medias_seguras)
X_teste_limpo[colunas_com_vazios] = X_teste_limpo[colunas_com_vazios].fillna(medias_seguras)

In [9]:
# balanceamento de classes apenas no treino
smote = SMOTE(random_state=42)
X_treino_bal, y_treino_bal = smote.fit_resample(X_treino_limpo, y_treino)

# padronização para outliers (RobustScaler) trabalha com quatis
scaler_local = RobustScaler()
X_treino_esc = scaler_local.fit_transform(X_treino_bal)
X_teste_esc = scaler_local.transform(X_teste_limpo)

In [10]:
#verificar acuracia local
knn_local = KNeighborsClassifier(n_neighbors=3)
knn_local.fit(X_treino_esc, y_treino_bal)
previsoes_locais = knn_local.predict(X_teste_esc)

print(f"⚖️ ACURÁCIA LOCAL (SMOTE + ROBUST): {accuracy_score(y_teste, previsoes_locais) * 100:.2f}%")
print("\n📊 RELATÓRIO DO MODELO LOCAL:")
print(classification_report(y_teste, previsoes_locais))

⚖️ ACURÁCIA LOCAL (SMOTE + ROBUST): 77.39%

📊 RELATÓRIO DO MODELO LOCAL:
              precision    recall  f1-score   support

           0       0.84      0.79      0.81        72
           1       0.68      0.74      0.71        43

    accuracy                           0.77       115
   macro avg       0.76      0.77      0.76       115
weighted avg       0.78      0.77      0.78       115



In [11]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import RepeatedStratifiedKFold, cross_validate
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

# 1. Configurando as Features (A sua lista original de 6 colunas)
FEATURE_COLS = ['Pregnancies', 'Glucose', 'BloodPressure', 'BMI', 'DiabetesPedigreeFunction', 'Age']
TARGET = 'Outcome'

df_train = pd.read_csv('diabetes_dataset.csv')

# Tratando os zeros como NaN para o Imputer trabalhar
VITALS = ['Glucose', 'BloodPressure', 'BMI']
for col in VITALS:
    df_train[col] = pd.to_numeric(df_train[col], errors='coerce').replace(0, np.nan)

X_train = df_train[FEATURE_COLS]
y_train = df_train[TARGET]

# ==========================================
# 2. O SEU PIPELINE (A Estratégia Elegante)
# ==========================================
# Olha como é mais limpo: Tapar buraco -> Escalonar -> Balancear -> Prever
seu_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('smote', SMOTE(random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=3))
])

# ==========================================
# 3. O MOTOR DE TESTE EXAUSTIVO (500 Rodadas)
# ==========================================
cv_strategy = RepeatedStratifiedKFold(n_splits=10, n_repeats=50, random_state=42)

cv_results = cross_validate(
    seu_pipeline,
    X_train,
    y_train,
    cv=cv_strategy,
    scoring=['accuracy', 'recall'],
    n_jobs=-1
)

acc_mean = cv_results['test_accuracy'].mean() * 100
acc_std = cv_results['test_accuracy'].std() * 100
recall_mean = cv_results['test_recall'].mean() * 100
recall_std = cv_results['test_recall'].std() * 100

print('\n' + '='*40)
print('RESULTADO DA SUA ESTRATÉGIA (500x)')
print('='*40)
print(f'Acurácia Média: {acc_mean:.2f}% (+/- {acc_std:.2f}%)')
print(f'Recall Médio:   {recall_mean:.2f}% (+/- {recall_std:.2f}%)')
print('='*40)

Iniciando O GRANDE DUELO (Sua Estratégia em 500 rodadas)...

🏆 RESULTADO DA SUA ESTRATÉGIA (500x)
Acurácia Média: 70.47% (+/- 5.96%)
Recall Médio:   69.76% (+/- 9.36%)
